## Planning:
- Load data
- Check variable and target
- Check correlation between numeric variable

In [1]:
# data processing
import pandas as pd
import numpy as np

# data visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# data modeling 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# model 
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submit = pd.read_csv("sample_submission.csv")

In [4]:
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(train['Arrival Delay in Minutes'].mean())

In [5]:
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(test['Arrival Delay in Minutes'].mean())

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [7]:
non_numeric_cols = train.select_dtypes(exclude='number').columns.tolist()

In [9]:
oof_logit_1 = pd.read_csv("blending\oof_logit_1_5cv.csv")
oof_xgb_2 = pd.read_csv("blending\oof_xgb_2_5cv.csv")
oof_xgb_6 = pd.read_csv("blending\oof_xgb_6_5cv_optuna.csv")
oof_xgb_7 = pd.read_csv("blending\oof_xgb_7_20cv_optuna.csv")

In [11]:
sub_logit_1 = pd.read_csv("blending\sub_logit_1.csv")
sub_xgb_2 = pd.read_csv("blending\sub_xgb_2.csv")
sub_xgb_6 = pd.read_csv("blending\sub_xgb_6_5cv_optuna.csv")
sub_xgb_7 = pd.read_csv("blending\sub_xgb_7_20cv_optuna.csv")

In [59]:
sub_logit_1.rename(columns={"satisfaction": oof_logit_1.columns.to_list()[2]}, inplace=True)

In [61]:
sub_xgb_2.rename(columns={"satisfaction": oof_xgb_2.columns.to_list()[2]}, inplace=True)
sub_xgb_6.rename(columns={"satisfaction": oof_xgb_6.columns.to_list()[2]}, inplace=True)
sub_xgb_7.rename(columns={"satisfaction": oof_xgb_7.columns.to_list()[2]}, inplace=True)

## Basic Data Process for Modeling

Data preparation - Create Dummy variable

In [12]:
# train_dummy = pd.get_dummies(train, columns=non_numeric_cols, dtype=int)
train_dummy = pd.get_dummies(train, columns=non_numeric_cols,drop_first=True, dtype=int)

In [13]:
test_dummy = pd.get_dummies(test, columns=non_numeric_cols.remove('satisfaction'),drop_first=True, dtype=int)

In [14]:
'''
X = train_dummy.drop(columns=["id","satisfaction_True"])
y = train_dummy["satisfaction_True"]
X_test = test_dummy.drop(columns=["id"])
'''

'\nX = train_dummy.drop(columns=["id","satisfaction_True"])\ny = train_dummy["satisfaction_True"]\nX_test = test_dummy.drop(columns=["id"])\n'

In [15]:
train_data = train_dummy[["id","satisfaction_True"]]

In [27]:
train_data = pd.concat([train_data,oof_logit_1.iloc[:,[2]]],axis=1)

In [29]:
train_data = pd.concat([train_data,oof_xgb_2.iloc[:,[2]],oof_xgb_6.iloc[:,[2]],oof_xgb_7.iloc[:,[2]]],axis=1)

In [65]:
train_data.head(2)

,id,satisfaction_True,oof_logit_1_5cv,oof_xgb_2_5cv,oof_xgb_6_5cv_optuna,oof_xgb_7_20cv_optuna
0,0,0,0.089254,0.264970,0.257997,0.232633
1,1,1,0.931650,0.957369,0.957024,0.950261


In [62]:
test_data = test_dummy[["id"]]

In [63]:
test_data = pd.concat([test_data,sub_logit_1.iloc[:,[1]],sub_xgb_2.iloc[:,[1]],sub_xgb_6.iloc[:,[1]],sub_xgb_7.iloc[:,[1]]],axis=1)

In [64]:
test_data.head(2)

,id,oof_logit_1_5cv,oof_xgb_2_5cv,oof_xgb_6_5cv_optuna,oof_xgb_7_20cv_optuna
0,699635,0.089525,0.236491,0.230871,0.229289
1,699636,0.582690,0.899090,0.890153,0.891726


In [70]:
X = train_data.drop(columns=["id","satisfaction_True"])
y = train_data["satisfaction_True"]
X_test = test_data.drop(columns=["id"])

## Find Optimum Parameter using Optuna for XGBOOST

## Model using XGboost 5 fold CV

In [71]:
SEED = 42
N_FOLDS = 5
TARGET = 'satisfaction_True'

In [72]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [73]:
def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [67]:
final_params = {
    # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,
        # early stopping round
        "early_stopping_rounds": 200,

        "learning_rate": 0.023621412607945612,
        "max_depth": 8,
        "reg_lambda": 0.15767822496360231,
        "alpha": 0.3574854835852679,
        "min_child_weight":29
}

In [74]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(**final_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [75]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.95903
  fold 1: AUC = 0.95800
  fold 2: AUC = 0.95976
  fold 3: AUC = 0.95908
  fold 4: AUC = 0.95922
XGBoost | OOF AUC = 0.95167 (mean 0.95902 ± 0.00057)


In [76]:
def save_sub(pred, name):
    sub = submit.copy()
    sub['satisfaction'] = pred
    sub.to_csv(name, index=False)
    print('saving file:', name)

In [77]:
save_sub(test_xgb,'sub_xgb_8_5cv_stacking.csv')

saving file: sub_xgb_8_5cv_stacking.csv


### XGB 20 Folds CV

In [78]:
SEED = 42
N_FOLDS = 20
TARGET = 'satisfaction_True'

In [79]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [80]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.95958
  fold 1: AUC = 0.95879
  fold 2: AUC = 0.95942
  fold 3: AUC = 0.95870
  fold 4: AUC = 0.95654
  fold 5: AUC = 0.95829
  fold 6: AUC = 0.95577
  fold 7: AUC = 0.96163
  fold 8: AUC = 0.95855
  fold 9: AUC = 0.96084
  fold 10: AUC = 0.95888
  fold 11: AUC = 0.96090
  fold 12: AUC = 0.95840
  fold 13: AUC = 0.95788
  fold 14: AUC = 0.95944
  fold 15: AUC = 0.96072
  fold 16: AUC = 0.96030
  fold 17: AUC = 0.95810
  fold 18: AUC = 0.95915
  fold 19: AUC = 0.95954
XGBoost | OOF AUC = 0.94514 (mean 0.95907 ± 0.00140)


In [81]:
# submission
save_sub(test_xgb,'sub_xgb_9_20cv_stacking.csv')

saving file: sub_xgb_9_20cv_stacking.csv


In [71]:
'''
oof_xgb_df = train[['id']].copy()
oof_xgb_df.loc[:,'oof_xgb_7_20cv_optuna'] = oof_xgb
oof_xgb_df.to_csv('oof_xgb_7_20cv_optuna.csv')
'''

In [82]:
auc_xgb

0.945144978672748